# baseline TASK-001 · 기존 결과 점검과 생성 Accuracy

기준: 첨부 baseline v3 실행본. 모델·프롬프트·384² 픽셀 예산·greedy 2토큰 유지.
**재설치, 재학습, 전체 test 추론을 실행하는 셀은 없습니다.** 원본 Notebook과 기존 어댑터/제출물은 보존하세요.

1. 기존 `AI2_Challenge` 폴더에 이 파일을 놓고 기존 baseline 커널에서 실행합니다.
2. 우선 기본 설정 그대로 위에서 아래로 실행하여 환경·분할 점검 결과를 확인합니다. `RUN_SMOKE=False`이면 모델은 로드하지 않습니다.
3. 원본 학습 당시 train.csv·분할·어댑터 연결을 확인한 뒤 설정 셀의 계보 필드를 채우고 `RUN_SMOKE=True`로 5~10개만 실행합니다.
4. 소량 결과를 확인한 다음에만 `RUN_FULL_COMPARISON=True`로 같은 20문항 LoRA/base 비교를 실행합니다.

현재 생성 Accuracy는 미측정. 0.70300은 사용자 보고값이며 이 실행본·제출 파일과의 연결은 미확인입니다.
기존 23셀의 위치를 가능한 한 유지하고 설치/학습 셀을 점검/평가 준비 셀로 교체했습니다. 저장 출력은 새 결과로 오인하지 않도록 비웠습니다.

## 1. 실행 설정 · 먼저 점검만 수행

In [1]:
from pathlib import Path
import os, json, hashlib, platform, importlib.metadata as metadata
from datetime import datetime, timezone
import uuid, re, time, traceback

TASK_ID = "TASK-001"
SEED = 42
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 2  # 원본 generate의 실제 값. 선언만 되어 있던 8로 바꾸지 않음.
PROJECT_ROOT = Path.cwd()
BASELINE_NOTEBOOK_PATH = PROJECT_ROOT / "(260902)_baseline_desktop5060ti_offline(1).ipynb"
TASK_NOTEBOOK_PATH = PROJECT_ROOT / "baseline_TASK-001.ipynb"
RUN_SMOKE = False
RUN_FULL_COMPARISON = False
SMOKE_N = 5  # 5~10. 같은 검증 목록의 앞부분, 정오를 보고 골라내지 않음.

# 현재 해시를 복사하는 것만으로 과거 학습 데이터와의 일치가 증명되지는 않습니다.
# 학습 당시 파일/실행 기록으로 확인한 값과 근거를 입력하세요.
TRAIN_CSV_AT_TRAINING_SHA256 = None
TRAINING_LINEAGE_CONFIRMED = False
TRAINING_LINEAGE_NOTE = ""  # 이 CSV의 seed=42 200→180/20으로 이 어댑터를 학습했다는 근거
SCORE_070300_LINK_CONFIRMED = False
SCORE_LOCATION_OR_SUBMISSION_ID = ""  # Public/로컬 구분과 해당 제출 ID 등
NEAR_DUPLICATES_REVIEWED = False
NEAR_DUPLICATE_REVIEW_NOTE = ""  # 유사 이미지 후보가 있을 때 검토 근거

# 별도 파일 쓰기 전에도 사용할 수 있는 읽기 전용 함수
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def file_info(path, hash_content=True):
    p = Path(path)
    if not p.is_file():
        return {"path": str(p.resolve()), "exists": False}
    return {"path": str(p.resolve()), "exists": True, "size_bytes": p.stat().st_size,
            "mtime_ns": p.stat().st_mtime_ns,
            "sha256": sha256_file(p) if hash_content else None}

In [2]:
ASSET_DIR = PROJECT_ROOT / "downloads"
LIB_DIR = ASSET_DIR / "libs"
MODEL_DIR = ASSET_DIR / "models" / "Qwen2.5-VL-3B-Instruct"
MODEL_ID = str(MODEL_DIR)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
SAVE_DIR = OUTPUT_DIR / "qwen2_5_vl_3b_lora"
BASELINE_SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

preflight = {"task_id": TASK_ID, "checked_at": datetime.now(timezone.utc).isoformat(),
             "python": platform.python_version(), "platform": platform.platform(),
             "packages": {}, "paths": {}, "csv": {}, "gpu": None,
             "baseline_notebook": file_info(BASELINE_NOTEBOOK_PATH),
             "task_notebook": file_info(TASK_NOTEBOOK_PATH),
             "baseline_submission": file_info(BASELINE_SUBMISSION_PATH)}
for package in ["torch", "torchvision", "transformers", "accelerate", "peft",
                "bitsandbytes", "pandas", "Pillow", "numpy"]:
    try:
        preflight["packages"][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        preflight["packages"][package] = None
for name, path in [("libs", LIB_DIR), ("model", MODEL_DIR), ("data", DATA_DIR), ("adapter", SAVE_DIR)]:
    preflight["paths"][name] = {"path": str(path.resolve()), "exists": path.is_dir()}
try:
    import pandas as pd
    for name in ["train.csv", "test.csv", "sample_submission.csv"]:
        p = DATA_DIR / name
        if p.is_file():
            df_check = pd.read_csv(p)
            preflight["csv"][name] = {**file_info(p), "rows": len(df_check), "columns": list(df_check.columns)}
        else:
            preflight["csv"][name] = {"exists": False}
except Exception as exc:
    preflight["csv_error"] = repr(exc)
print(json.dumps(preflight, ensure_ascii=False, indent=2))

{
  "task_id": "TASK-001",
  "checked_at": "2026-09-21T02:19:06.920933+00:00",
  "python": "3.11.9",
  "platform": "Windows-10-10.0.26200-SP0",
  "packages": {
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "transformers": "4.57.6",
    "accelerate": "1.15.0",
    "peft": "0.20.0",
    "bitsandbytes": "0.50.2",
    "pandas": "3.0.5",
    "Pillow": "12.3.0",
    "numpy": "2.4.6"
  },
  "paths": {
    "libs": {
      "path": "C:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\downloads\\libs",
      "exists": true
    },
    "model": {
      "path": "C:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\downloads\\models\\Qwen2.5-VL-3B-Instruct",
      "exists": true
    },
    "data": {
      "path": "C:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\data",
      "exists": true
    },
    "adapter": {
      "path": "C:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\output\\qwen2_5_vl_3b_lora",
      "exists": true
    }
  },
  "csv": {
    "train.csv": {
      "path": "C:\\Users\\SSAFY\\Desktop\\AI2_Ch

In [3]:
# 설치 없음: 기존 환경을 유지합니다. 원본 pip 셀은 이 점검본에서 실행하지 않습니다.

In [4]:
try:
    import torch
    preflight["cuda_available"] = torch.cuda.is_available()
    preflight["gpu"] = [
        {"index": i, "name": torch.cuda.get_device_name(i),
         "total_vram_bytes": torch.cuda.get_device_properties(i).total_memory}
        for i in range(torch.cuda.device_count())
    ]
except Exception as exc:
    preflight["cuda_available"] = False
    preflight["torch_error"] = repr(exc)
print(json.dumps({k: preflight.get(k) for k in ["cuda_available", "gpu", "torch_error"]},
                 ensure_ascii=False, indent=2))

{
  "cuda_available": true,
  "gpu": [
    {
      "index": 0,
      "name": "NVIDIA GeForce RTX 5060 Ti",
      "total_vram_bytes": 17102864384
    }
  ],
  "torch_error": null
}


In [5]:
# 어댑터는 내용 해시, 대형 기본 가중치는 크기/수정 시각을 기록합니다.
# 기본 가중치 내용 전체를 해시하지 않으므로 완전한 동일성 증명은 아닙니다.
preflight["adapter_files"] = [file_info(p) for p in sorted(SAVE_DIR.rglob("*")) if p.is_file()] if SAVE_DIR.is_dir() else []
preflight["base_files"] = [file_info(p, hash_content=p.suffix in {".json", ".txt", ".jinja"})
                           for p in sorted(MODEL_DIR.rglob("*")) if p.is_file()] if MODEL_DIR.is_dir() else []
print("원본 실행본 해시:", preflight["baseline_notebook"])
print("어댑터 파일 수:", len(preflight["adapter_files"]))
print("기본 모델 파일 수:", len(preflight["base_files"]))

원본 실행본 해시: {'path': 'C:\\Users\\SSAFY\\Desktop\\AI2_Challenge\\(260902)_baseline_desktop5060ti_offline(1).ipynb', 'exists': False}
어댑터 파일 수: 12
기본 모델 파일 수: 25


## 2. 기존 분할 복원과 실행 기록 준비

seed만으로 과거 학습 ID 일치를 확정하지 않습니다. `train.csv` 내용·순서와 학습 당시 파일의 연결을 확인해야 합니다.
점검 결과는 실행별 고유 폴더에 저장하며 `output/submission.csv`와 어댑터를 덮어쓰지 않습니다.
`split_manifest.csv`는 **현재 CSV에서 복원한 후보 분할**이며 계보가 확인되기 전에는 과거 실제 분할로 확정하지 않습니다.

In [6]:
import pandas as pd
from PIL import Image
Image.MAX_IMAGE_PIXELS = None  # 원본 이미지 로드 설정 유지
import random
random.seed(SEED)
if "torch" in globals():
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

RUN_DIR = OUTPUT_DIR / TASK_ID / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True, exist_ok=False)
def save_json(name, obj):
    (RUN_DIR / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
run_config = {**preflight, "run_dir": str(RUN_DIR.resolve()), "seed": SEED,
              "image_pixel_budget": IMAGE_SIZE ** 2,
              "generation": {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False},
              "quantization": {"load_in_4bit": True, "type": "nf4", "double_quant": True, "compute_dtype": "float16"},
              "training_lineage_confirmed": TRAINING_LINEAGE_CONFIRMED,
              "training_lineage_note": TRAINING_LINEAGE_NOTE,
              "train_csv_at_training_sha256": TRAIN_CSV_AT_TRAINING_SHA256,
              "reported_baseline_score": 0.70300,
              "score_link_confirmed": SCORE_070300_LINK_CONFIRMED,
              "score_location_or_submission_id": SCORE_LOCATION_OR_SUBMISSION_ID}
save_json("run_config.json", run_config)
save_json("metrics.json", {"status": "not_run", "EXP-001": None, "EXP-002": None})
required = {"id", "path", "question", "a", "b", "c", "d", "answer"}
train_all = pd.read_csv(DATA_DIR / "train.csv")  # 원본 baseline와 같은 dtype 추론·행 순서
assert required <= set(train_all.columns), f"필수 컬럼 누락: {required - set(train_all.columns)}"
assert len(train_all) >= 200, "원본 200행 샘플 복원 불가"
assert not train_all[list(required)].isna().any().any(), "필수 값 결측"
assert not train_all["id"].astype(str).duplicated().any(), "중복 ID"
assert train_all["answer"].astype(str).str.strip().str.lower().isin(list("abcd")).all(), "허용되지 않은 정답"
train_df = train_all.sample(n=200, random_state=SEED).reset_index(drop=True)
split = int(len(train_df) * 0.9)
train_subset, valid_subset = train_df.iloc[:split].copy(), train_df.iloc[split:].copy()
print("점검 파일:", RUN_DIR.resolve())
print("복원한 후보 분할:", len(train_subset), len(valid_subset))

점검 파일: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-001\20260921T021908Z_36a9022d
복원한 후보 분할: 180 20


## 3. 저장된 모델 로드 함수 · 호출 전에는 가중치를 읽지 않음

In [7]:
def load_saved_model():
    import torch
    from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig
    from peft import PeftModel, prepare_model_for_kbit_training
    assert torch.cuda.is_available(), "기존 GPU 환경에서 실행하세요. CPU 추론으로 자동 전환하지 않습니다."
    assert (SAVE_DIR / "adapter_config.json").is_file(), "저장된 LoRA 어댑터가 없습니다. 재학습하지 마세요."
    cfg = json.loads((SAVE_DIR / "adapter_config.json").read_text(encoding="utf-8"))
    assert str(cfg.get("peft_type")).upper() == "LORA", "기존 LoRA가 아닌 어댑터"
    assert cfg.get("r") == 8 and cfg.get("lora_alpha") == 16 and cfg.get("lora_dropout") == 0.05, "어댑터 설정이 인계와 다름"
    assert cfg.get("bias") == "none" and not cfg.get("modules_to_save"), "base 비활성 비교 조건 재검토 필요"
    expected_targets = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    assert set(cfg.get("target_modules", [])) == expected_targets, "LoRA target_modules가 기준과 다름"
    # 원본과 같이 MODEL_DIR processor 사용. 저장 processor 파일도 해시 기록되므로 변경 여부 검토 가능.
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=IMAGE_SIZE**2,
        max_pixels=IMAGE_SIZE**2, trust_remote_code=True, local_files_only=True)
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    base_model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True, local_files_only=True)
    # 원본 prepare가 수행하는 freeze/cast를 유지하되 평가에 불필요한 checkpointing은 끔.
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=False)
    model = PeftModel.from_pretrained(base_model, str(SAVE_DIR), is_trainable=False, local_files_only=True)
    model.eval()
    assert not model.training and not any(p.requires_grad for p in model.parameters())
    run_config["adapter_config"] = cfg
    run_config["processor_class"] = type(processor).__name__
    run_config["image_processor_class"] = type(processor.image_processor).__name__
    run_config["image_processor_config"] = processor.image_processor.to_dict()
    run_config["tokenizer_class"] = type(processor.tokenizer).__name__
    run_config["effective_generation_config"] = model.generation_config.to_dict()
    run_config["device_map"] = {k: str(v) for k, v in getattr(model, "hf_device_map", {}).items()}
    save_json("run_config.json", run_config)
    return model, processor

## 4. 프롬프트 · 원본 유지

In [8]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

## 5. 정답을 포함하지 않는 평가 입력

In [9]:
# 원본 VQAMCDataset의 사용자/시스템 메시지 구성과 동일. answer는 이 함수에서 읽지 않음.
def build_eval_messages(row, img):
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [{"type": "image", "image": img},
                                     {"type": "text", "text": user_text}]}
    ]

## 6. 분할 이미지 감사 · 동일 경로/파일/픽셀과 유사 이미지 후보

In [10]:
def dhash_rgb(img):
    pixels = list(img.convert("L").resize((9, 8)).getdata())
    return sum(int(pixels[y*9+x] > pixels[y*9+x+1]) << (y*8+x) for y in range(8) for x in range(8))

def audit_split(train_subset, valid_subset, data_dir):
    rows = []
    for split_name, frame in [("train", train_subset), ("valid", valid_subset)]:
        for position, (_, row) in enumerate(frame.iterrows()):
            path = Path(data_dir) / str(row["path"])
            record = {"id": str(row["id"]), "split": split_name, "split_position": position,
                      "image_path": str(path.resolve()), "file_sha256": None, "pixel_sha256": None,
                      "dhash": None, "image_error": None}
            try:
                record["file_sha256"] = sha256_file(path)
                with Image.open(path) as im:
                    img = im.convert("RGB")
                    record["pixel_sha256"] = hashlib.sha256(str(img.size).encode() + img.tobytes()).hexdigest()
                    record["dhash"] = f"{dhash_rgb(img):016x}"
            except Exception as exc:
                record["image_error"] = repr(exc)
            rows.append(record)
    manifest = pd.DataFrame(rows)
    exact = []
    for key in ["id", "image_path", "file_sha256", "pixel_sha256"]:
        for value, group in manifest.dropna(subset=[key]).groupby(key):
            if group["split"].nunique() > 1:
                exact.append({"kind": key, "value": str(value), "ids": group["id"].tolist()})
    near = []
    for a in manifest[manifest.split == "train"].to_dict("records"):
        for b in manifest[manifest.split == "valid"].to_dict("records"):
            if a["dhash"] is not None and b["dhash"] is not None:
                distance = (int(a["dhash"],16) ^ int(b["dhash"],16)).bit_count()
                if distance <= 4 and a["pixel_sha256"] != b["pixel_sha256"]:
                    near.append({"train_id": a["id"], "valid_id": b["id"], "dhash_distance": distance})
    return manifest, {"exact_cross_split_duplicates": exact, "near_duplicate_candidates": near,
                      "image_errors": manifest[manifest.image_error.notna()].to_dict("records"),
                      "near_duplicate_method": "64-bit dHash Hamming<=4; 후보 탐색이며 누수 확정/완전 탐지 아님"}

manifest, split_audit = audit_split(train_subset, valid_subset, DATA_DIR)
manifest["train_csv_sha256"] = sha256_file(DATA_DIR / "train.csv")
manifest["historical_split_confirmed"] = bool(TRAINING_LINEAGE_CONFIRMED and TRAINING_LINEAGE_NOTE.strip()
    and TRAIN_CSV_AT_TRAINING_SHA256 == manifest["train_csv_sha256"].iloc[0])
manifest.to_csv(RUN_DIR / "split_manifest.csv", index=False, encoding="utf-8-sig")
save_json("split_audit.json", split_audit)
print(json.dumps(split_audit, ensure_ascii=False, indent=2))
print("과거 학습 분할 연결:", bool(manifest.historical_split_confirmed.all()))
print("주의: dHash는 크롭/텍스트 차이/촬영 세션을 모두 판별하지 못합니다.")

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_29036\356702568.py:2: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(img.convert("L").resize((9, 8)).getdata())
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_29036\356702568.py:2: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(img.convert("L").resize((9, 8)).getdata())


{
  "exact_cross_split_duplicates": [],
  "near_duplicate_candidates": [],
  "image_errors": [],
  "near_duplicate_method": "64-bit dHash Hamming<=4; 후보 탐색이며 누수 확정/완전 탐지 아님"
}
과거 학습 분할 연결: False
주의: dHash는 크롭/텍스트 차이/촬영 세션을 모두 판별하지 못합니다.


## 7. 학습 대신 독립성 확인 · 기존 어댑터 활용

In [11]:
# 원본 학습 셀을 실행하지 않습니다. 학습 루프/loss/체크포인트를 변경하지 않음.
def assert_evaluation_ready():
    assert TRAINING_LINEAGE_CONFIRMED and TRAINING_LINEAGE_NOTE.strip(), "학습 당시 CSV·분할과 이 어댑터의 연결 근거가 필요합니다."
    assert TRAIN_CSV_AT_TRAINING_SHA256 == sha256_file(DATA_DIR / "train.csv"), "학습 당시 CSV 해시 연결 미확인/불일치"
    assert not split_audit["image_errors"], "손상/누락 이미지 해결 후 실행"
    assert not split_audit["exact_cross_split_duplicates"], "분할 간 동일 이미지/ID 발견: 00에 분할 검토 요청"
    if split_audit["near_duplicate_candidates"]:
        assert NEAR_DUPLICATES_REVIEWED and NEAR_DUPLICATE_REVIEW_NOTE.strip(), "유사 이미지 후보를 검토한 뒤 실행"
    assert 5 <= SMOKE_N <= 10
    run_config["training_lineage_confirmed"] = True
    run_config["training_lineage_note"] = TRAINING_LINEAGE_NOTE
    run_config["train_csv_at_training_sha256"] = TRAIN_CSV_AT_TRAINING_SHA256
    run_config["near_duplicate_review"] = {"reviewed": NEAR_DUPLICATES_REVIEWED, "note": NEAR_DUPLICATE_REVIEW_NOTE}
    save_json("run_config.json", run_config)

## 8. 생성 부분 추출·파서·평가 함수

In [12]:
# 비교를 위한 원본 파서를 그대로 보존.
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last
    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

def parse_choice(text):
    s = text.strip().lower()
    if not s:
        return None, "empty"
    # 전체 응답이 명확한 단일 선지 형식일 때만 허용. 문장에서 임의 글자 추출 금지.
    match = re.fullmatch(r"(?:([a-d])|\(([a-d])\)|\[([a-d])\])[.!。]?", s)
    if match:
        return next(v for v in match.groups() if v), "ok"
    candidates = set(re.findall(r"(?<![a-z0-9_])[a-d](?![a-z0-9_])", s))
    return None, "multiple_candidates" if len(candidates) > 1 else "unrecognized"

def summarize_predictions(frame, elapsed_seconds, memory):
    n = len(frame)
    assert n > 0
    return {"n": n, "correct": int(frame.correct.sum()), "accuracy": float(frame.correct.sum()/n),
            "parsing_failures": int((frame.parse_status != "ok").sum()),
            "parsing_failure_rate": float((frame.parse_status != "ok").sum()/n),
            "legacy_full_accuracy": float(frame.legacy_full_correct.sum()/n),
            "legacy_generated_accuracy": float(frame.legacy_generated_correct.sum()/n),
            "extraction_changes": int((frame.legacy_full_choice != frame.legacy_generated_choice).sum()),
            "parser_changes": int((frame.legacy_generated_choice != frame.choice.fillna("<FAIL>")).sum()),
            "parser_correct_gained": int((~frame.legacy_generated_correct & frame.correct).sum()),
            "parser_correct_lost": int((frame.legacy_generated_correct & ~frame.correct).sum()),
            "possible_length_limit": int(frame.at_token_limit.sum()),
            "elapsed_seconds": elapsed_seconds, "cuda_memory": memory,
            "interpretation": "동일 소규모 홀드아웃. 파싱 실패를 분모에 포함."}

def evaluate_rows(model, processor, frame, exp_id, variant, phase):
    import torch
    from tqdm.auto import tqdm
    name = f"{phase}_{exp_id}_{variant}"
    log_path = RUN_DIR / f"{name}_raw.jsonl"
    assert not log_path.exists(), "동일 실행 내 덮어쓰기를 막습니다. 새 RUN_DIR에서 재실행하세요."
    devices = list(range(torch.cuda.device_count()))
    for i in devices:
        torch.cuda.synchronize(i)
        torch.cuda.reset_peak_memory_stats(i)
    start = time.perf_counter()
    records = []
    input_device = model.get_input_embeddings().weight.device
    try:
        model.eval()
        with log_path.open("x", encoding="utf-8") as log:
            for _, row in tqdm(frame.iterrows(), total=len(frame), desc=f"{phase} {exp_id} {variant}"):
                with Image.open(DATA_DIR / str(row["path"])) as im:
                    img = im.convert("RGB")
                messages = build_eval_messages(row, img)
                assert [m["role"] for m in messages] == ["system", "user"]
                text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = processor(text=[text], images=[img], return_tensors="pt").to(input_device)
                input_length = inputs["input_ids"].shape[1]
                with torch.inference_mode():
                    out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                             eos_token_id=processor.tokenizer.eos_token_id)
                generated_ids = out_ids[:, input_length:]
                raw_full = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
                raw_generated = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
                choice, parse_status = parse_choice(raw_generated)
                legacy_full = extract_choice(raw_full)
                legacy_generated = extract_choice(raw_generated)
                gold = str(row["answer"]).strip().lower()  # 정답은 생성 후 채점에만 사용
                token_ids = generated_ids[0].detach().cpu().tolist()
                reasons = []
                if legacy_full != legacy_generated:
                    reasons.append("input_tokens_removed")
                if legacy_generated != choice:
                    reasons.append("format_recognized" if parse_status == "ok" else "fallback_a_or_unsafe_parse_removed")
                rec = {"experiment_id": exp_id, "phase": phase, "variant": variant,
                       "id": str(row["id"]), "gold": gold, "input_tokens": int(input_length),
                       "prompt_sha256": hashlib.sha256(text.encode()).hexdigest(),
                       "raw_full": raw_full, "raw_generated": raw_generated,
                       "generated_token_ids": token_ids, "generated_tokens": len(token_ids),
                       "at_token_limit": len(token_ids) >= MAX_NEW_TOKENS,
                       "legacy_full_choice": legacy_full, "legacy_generated_choice": legacy_generated,
                       "choice": choice, "parse_status": parse_status, "change_reason": ";".join(reasons),
                       "legacy_full_correct": legacy_full == gold, "legacy_generated_correct": legacy_generated == gold,
                       "correct": choice == gold, "image_grid_thw": inputs["image_grid_thw"].detach().cpu().tolist()}
                records.append(rec)
                log.write(json.dumps(rec, ensure_ascii=False) + "\n")
                log.flush()
        for i in devices:
            torch.cuda.synchronize(i)
        elapsed = time.perf_counter() - start
        memory = [{"device": i, "max_allocated_bytes": torch.cuda.max_memory_allocated(i),
                   "max_reserved_bytes": torch.cuda.max_memory_reserved(i)} for i in devices]
        result = pd.DataFrame(records)
        result.to_csv(RUN_DIR / f"{name}_predictions.csv", index=False, encoding="utf-8-sig")
        metrics = summarize_predictions(result, elapsed, memory)
        save_json(f"{name}_metrics.json", metrics)
        return result, metrics
    except BaseException as exc:
        save_json(f"{name}_failure.json", {"status": "failed_or_interrupted", "error": repr(exc),
            "completed_samples": len(records), "elapsed_seconds": time.perf_counter()-start,
            "traceback": traceback.format_exc(), "partial_log": str(log_path)})
        raise

In [13]:
# 대표 형식 확인. 합성 입력의 검사이며 실제 모델 성능이 아닙니다.
parser_examples = ["b", "(b)", "b.", "B", "[c]", "", "a or b", "Answer: b", "banana", "(b"]
print(pd.DataFrame([{"text": s, "legacy": extract_choice(s), "new": parse_choice(s)[0],
                     "status": parse_choice(s)[1]} for s in parser_examples]).to_string(index=False))
if RUN_SMOKE:
    assert_evaluation_ready()
    try:
        if "model" in globals():
            del model
            import gc
            gc.collect()
            torch.cuda.empty_cache()
        model, processor = load_saved_model()
    except BaseException as exc:
        save_json("model_load_failure.json", {"status": "failed_or_interrupted", "error": repr(exc), "traceback": traceback.format_exc()})
        raise
    smoke_predictions, smoke_metrics = evaluate_rows(model, processor, valid_subset.iloc[:SMOKE_N],
                                                      "EXP-001", "lora", "smoke")
    smoke_run_dir = RUN_DIR.resolve()
    save_json("metrics.json", {"status": "smoke_completed_full_not_run", "EXP-001": smoke_metrics, "EXP-002": None})
    display(smoke_predictions[["id", "gold", "raw_generated", "legacy_full_choice", "legacy_generated_choice",
                               "choice", "parse_status", "correct", "at_token_limit"]])
    print(json.dumps(smoke_metrics, ensure_ascii=False, indent=2))
    print("20문항 추론 예상 시간(로드 제외):", smoke_metrics["elapsed_seconds"] / SMOKE_N * 20, "초/모델")
else:
    print("점검만 완료. 독립성·학습 이력 확인 후 RUN_SMOKE=True로 5~10문항을 실행하세요.")

     text legacy new              status
        b      b   b                  ok
      (b)      a   b                  ok
       b.      a   b                  ok
        B      b   b                  ok
      [c]      a   c                  ok
               a NaN               empty
   a or b      a NaN multiple_candidates
Answer: b      b NaN        unrecognized
   banana      a NaN        unrecognized
       (b      a NaN        unrecognized
점검만 완료. 독립성·학습 이력 확인 후 RUN_SMOKE=True로 5~10문항을 실행하세요.


## 9. 소량 결과 확인 후 실행하는 동일 20문항 비교
`RUN_FULL_COMPARISON=False`가 기본입니다. 이 비교는 기존 검증 20개를 유지하며 새 분할을 만들지 않습니다.
저장 LoRA를 다시 로드한 모델에서 LoRA 적용/비활성화를 순차 평가합니다. bias 학습·modules_to_save가 없는지 위에서 검사합니다.
20문항은 1문항당 5%p 변동합니다. 토큰 한도 도달은 잘림의 **후보**일 뿐 실제 잘림 확정이 아닙니다.

In [14]:
if RUN_FULL_COMPARISON:
    assert "smoke_predictions" in globals() and smoke_run_dir == RUN_DIR.resolve(), "이번 실행의 소량 결과부터 확인하세요."
    assert_evaluation_ready()
    lora_predictions, lora_metrics = evaluate_rows(model, processor, valid_subset, "EXP-001", "lora", "full")
    lora_layers = [m for m in model.modules() if hasattr(m, "lora_A") and hasattr(m, "disable_adapters")]
    assert lora_layers and all(not m.disable_adapters for m in lora_layers), "LoRA 활성 상태 확인 실패"
    with model.disable_adapter():
        assert all(m.disable_adapters for m in lora_layers), "LoRA 비활성화 확인 실패"
        base_predictions, base_metrics = evaluate_rows(model, processor, valid_subset, "EXP-002", "base_disabled", "full")
    assert all(not m.disable_adapters for m in lora_layers), "LoRA 복원 확인 실패"
    assert lora_predictions.id.tolist() == base_predictions.id.tolist()
    assert lora_predictions.prompt_sha256.tolist() == base_predictions.prompt_sha256.tolist()
    assert lora_predictions.image_grid_thw.tolist() == base_predictions.image_grid_thw.tolist()
    validation_predictions = pd.concat([lora_predictions, base_predictions], ignore_index=True)
    validation_predictions.to_csv(RUN_DIR / "validation_predictions.csv", index=False, encoding="utf-8-sig")
    save_json("metrics.json", {"status": "full_completed", "EXP-001": lora_metrics, "EXP-002": base_metrics,
                              "comparison_mode": "saved LoRA enabled vs disabled, same quantized base",
                              "accuracy_difference_lora_minus_base": lora_metrics["accuracy"]-base_metrics["accuracy"]})
    print(json.dumps({"EXP-001": lora_metrics, "EXP-002": base_metrics}, ensure_ascii=False, indent=2))
else:
    print("전체 20문항 비교: 실행 전")

전체 20문항 비교: 실행 전


## 10. 기존 제출 CSV 검사 · test 추론 없음
sample_submission의 컬럼·ID 순서를 기준으로 검사합니다. 정렬이 다르면 원본을 보존하고 정렬 사본만 별도 저장합니다.
파싱 실패를 임의 정답으로 대체하지 않습니다. 기존 CSV 검사로 제출 점수나 모델 성능을 증명하지 않습니다.

In [15]:
def validate_submission(sample, predictions, test=None):
    expected = list(sample.columns)
    if set(expected) != {"id", "answer"}:
        raise ValueError(f"제공 안내와 다른 sample_submission 컬럼: {expected}. 실제 규격 검토 필요")
    if list(predictions.columns) != expected:
        raise ValueError("sample_submission과 제출 컬럼/순서 불일치")
    for label, frame in [("sample", sample), ("predictions", predictions)]:
        if frame["id"].isna().any() or frame["id"].astype(str).duplicated().any():
            raise ValueError(f"{label}: 결측/중복 ID")
    if len(sample) != len(predictions) or set(sample.id.astype(str)) != set(predictions.id.astype(str)):
        raise ValueError("제출 ID 누락/추가 또는 행 수 불일치")
    if predictions.answer.isna().any() or not predictions.answer.isin(list("abcd")).all():
        raise ValueError("미해결/허용되지 않은 정답 값. 기본값 a로 채우지 마세요.")
    if test is not None:
        if test.id.isna().any() or test.id.astype(str).duplicated().any():
            raise ValueError("test ID 결측/중복")
        if len(test) != len(sample) or set(test.id.astype(str)) != set(sample.id.astype(str)):
            raise ValueError("test와 sample_submission ID 불일치")
    ordered = predictions.copy()
    ordered["id"] = ordered.id.astype(str)
    ordered = ordered.set_index("id").loc[sample.id.astype(str)].reset_index()[expected]
    return ordered, {"status": "passed", "rows": len(ordered),
                     "input_order_matches_sample": predictions.id.astype(str).tolist() == sample.id.astype(str).tolist(),
                     "test_ids_checked": test is not None}

sample_path = DATA_DIR / "sample_submission.csv"
if sample_path.is_file() and BASELINE_SUBMISSION_PATH.is_file():
    try:
        sample_submission = pd.read_csv(sample_path, dtype={"id": str, "answer": str})
        baseline_submission = pd.read_csv(BASELINE_SUBMISSION_PATH, dtype={"id": str, "answer": str})
        test_for_check = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str}) if (DATA_DIR / "test.csv").is_file() else None
        ordered, submission_check = validate_submission(sample_submission, baseline_submission, test_for_check)
        if not submission_check["input_order_matches_sample"]:
            ordered.to_csv(RUN_DIR / "baseline_submission_reordered.csv", index=False)
        save_json("submission_check.json", submission_check)
        print(submission_check)
    except Exception as exc:
        save_json("submission_check.json", {"status": "failed", "error": repr(exc)})
        raise
else:
    save_json("submission_check.json", {"status": "not_checked", "reason": "sample_submission/기존 submission 파일 없음"})
    print("기존 제출 검사 미수행: sample_submission 또는 기존 submission 파일이 없습니다.")

{'status': 'passed', 'rows': 6714, 'input_order_matches_sample': True, 'test_ids_checked': True}


## 11. 01에 돌려줄 출력

- 첫 점검: 출력 폴더의 `run_config.json`, `split_manifest.csv`, `split_audit.json`, `submission_check.json`.
- 소량 추론 후: `smoke_EXP-001_lora_predictions.csv`, `smoke_EXP-001_lora_metrics.json`, 원문 JSONL.
- 실패 시: 해당 `*_failure.json`과 마지막 오류 출력. 실행 도중 Notebook도 출력과 함께 저장하세요.
- 전체 비교는 첫 소량 결과를 확인한 후 별도로 실행합니다. 성공 시 `validation_predictions.csv`, `metrics.json` 추가.
- 0.70300이 나온 평가 위치/제출 ID와 기존 CSV·어댑터 연결 여부도 알려주세요.

로그에는 질문·선지·응답 원문과 로컬 경로가 포함됩니다. 학습 가중치는 업로드할 필요가 없습니다.
02 독립 검토 및 00 채택 판단은 아직 수행되지 않았습니다.